In [9]:
from google.colab import drive
drive.mount('/content/drive')

# 👇 cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"

!pwd
!ls
!ls data/raw



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml
data  scripts
 CleanData_jotpars.csv	'Job opportunities.xlsx'
 Dataset_jotpars.csv	 jobs.csv


In [10]:
from pathlib import Path
import re
from collections import Counter
import pandas as pd

# --- IMPORTANT: base path handling for both Colab and normal Python ---
try:
    # When running as a normal .py file (e.g. python scripts/extract_raw_skills.py)
    BASE_DIR = Path(__file__).resolve().parents[1]
except NameError:
    # When running inside a notebook / Colab cell
    BASE_DIR = Path.cwd()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def normalize_text(text: str) -> str:
    """Lowercase, remove weird chars, and collapse spaces."""
    if not isinstance(text, str):
        return ""

    # replace unicode dashes with space
    text = re.sub(r"[\u2010-\u2015\-]+", " ", text)
    # keep letters, digits, basic symbols, separators
    text = re.sub(r"[^A-Za-z0-9#+./ ,/;|]", " ", text)
    # collapse spaces
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()


def split_tokens(text: str):
    """Split a normalized string into tokens on comma/semicolon/slash/pipe."""
    if not text:
        return []
    parts = re.split(r"[;,/|]", text)
    tokens = []
    for p in parts:
        tok = re.sub(r"\s+", " ", p).strip()
        if tok:
            tokens.append(tok)
    return tokens


def add_from_series(series, bag):
    for val in series:
        norm = normalize_text(str(val))
        for tok in split_tokens(norm):
            bag.append(tok)


def main():
    all_tokens = []

    # 1) jobs.csv  (taglist)
    jobs_path = RAW_DIR / "jobs.csv"
    if jobs_path.exists():
        print(f"Reading {jobs_path}")
        jobs = pd.read_csv(jobs_path, encoding="utf-8", low_memory=False)
        col = "taglist"
        if col in jobs.columns:
            add_from_series(jobs[col], all_tokens)
        else:
            print(f"WARNING: column '{col}' not found in jobs.csv")
    else:
        print("WARNING: jobs.csv not found")

    # 2) Dataset_jotpars.csv  (requirment)
    ds_path = RAW_DIR / "Dataset_jotpars.csv"
    if ds_path.exists():
        print(f"Reading {ds_path}")
        ds = pd.read_csv(ds_path, encoding="latin1", low_memory=False)
        col = "requirment"
        if col in ds.columns:
            add_from_series(ds[col], all_tokens)
        else:
            print(f"WARNING: column '{col}' not found in Dataset_jotpars.csv")
    else:
        print("WARNING: Dataset_jotpars.csv not found")

    # 3) CleanData_jotpars.csv  (r)
    clean_path = RAW_DIR / "CleanData_jotpars.csv"
    if clean_path.exists():
        print(f"Reading {clean_path}")
        clean = pd.read_csv(clean_path, encoding="latin1", low_memory=False)
        col = "r"
        if col in clean.columns:
            add_from_series(clean[col], all_tokens)
        else:
            print(f"WARNING: column '{col}' not found in CleanData_jotpars.csv")
    else:
        print("WARNING: CleanData_jotpars.csv not found")

    # 4) Job opportunities.xlsx  (Required Skills)
    jobops_path = RAW_DIR / "Job opportunities.xlsx"
    if jobops_path.exists():
        print(f"Reading {jobops_path}")
        jobops = pd.read_excel(jobops_path, engine="openpyxl")
        col = "Required Skills"
        if col in jobops.columns:
            add_from_series(jobops[col], all_tokens)
        else:
            print(f"WARNING: column '{col}' not found in Job opportunities.xlsx")
    else:
        print("WARNING: Job opportunities.xlsx not found")

    # Count frequencies
    counter = Counter(all_tokens)
    print(f"Extracted {len(counter)} unique raw tokens.")

    # Simple noise filter: drop very short and obviously useless tokens
    stopwords = {
        "experience", "year", "years", "team", "environment", "project",
        "projects", "etc", "etc.", "knowledge", "ability", "skills",
        "english", "language", "management", "good", "strong"
    }

    rows = []
    for token, freq in counter.items():
        if token in stopwords:
            continue
        if len(token) <= 2 and token not in {"c", "go", "r"}:
            continue
        rows.append({"token": token, "frequency": freq})

    df = pd.DataFrame(rows).sort_values("frequency", ascending=False)
    out_path = PROCESSED_DIR / "raw_skill_tokens.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved {len(df)} tokens to {out_path}")


if __name__ == "__main__":
    main()


Reading /content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/data/raw/jobs.csv
Reading /content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/data/raw/Dataset_jotpars.csv
Reading /content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/data/raw/CleanData_jotpars.csv
Reading /content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/data/raw/Job opportunities.xlsx
Extracted 4750 unique raw tokens.
Saved 4688 tokens to /content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/data/processed/raw_skill_tokens.csv
